## ANNOTATIONS ANNOTATIONS

In [21]:
import os
import json
import numpy as np

from scipy.optimize import linear_sum_assignment
from pymongo import MongoClient

m2 = 0
a2 = 0

def iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    interW = max(0, xB - xA)
    interH = max(0, yB - yA)
    interArea = interW * interH

    if interArea == 0:
        return 0.0

    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    unionArea = boxAArea + boxBArea - interArea
    return interArea / unionArea


def best_iou_matching(boxesA, boxesB, threshold=0.4):
    if not boxesA or not boxesB:
        return 0

    nA, nB = len(boxesA), len(boxesB)
    iou_matrix = np.zeros((nA, nB))

    for i in range(nA):
        for j in range(nB):
            iou_matrix[i, j] = iou(boxesA[i], boxesB[j])

    row_ind, col_ind = linear_sum_assignment(-iou_matrix)
    matches = sum(iou_matrix[i, j] >= threshold for i, j in zip(row_ind, col_ind))
    return matches


def compute_inter_annotator_agreement(annotation_dir, iou_threshold=0.4):

    annotators = []
    annotator_names = []
    for file in os.listdir(annotation_dir):
        if file.endswith(".json"):
            with open(os.path.join(annotation_dir, file), "r") as f:
                data = json.load(f)
                annotators.append(data["annotations"])
                annotator_names.append(file)
                
    if len(annotators) < 2:
        raise ValueError("Need at least two annotators to compute agreement.")

    video_folders = sorted(set(a["videoFolder"] for ann in annotators for a in ann))
    print(video_folders)
    per_sample = []

    for folder in video_folders:
        sample_boxes = []
        for ann in annotators:
            boxes = []
            for entry in ann:
                if entry["videoFolder"] == folder:
                    boxes = [g["bbox"] for g in entry["groups"] if g["confidence"] >= 1]
                    break
            sample_boxes.append(boxes)

        #print(sample_boxes)

        global m2
        global a2
        pair_agreements = []
        for i in range(len(sample_boxes)):
            for j in range(i + 1, len(sample_boxes)):
                boxesA, boxesB = sample_boxes[i], sample_boxes[j]
                #print(boxesA)
                #print(boxesB)
                matches = best_iou_matching(boxesA, boxesB, iou_threshold)
                m2+=matches
                avg_boxes = (len(boxesA) + len(boxesB)) / 2 if (len(boxesA) + len(boxesB)) > 0 else 1
                pair_agreements.append(matches / avg_boxes)
                a2+=avg_boxes
        
        #print(pair_agreements)
        agreement = np.mean(pair_agreements) if pair_agreements else 1.0
        if agreement == 0 and sample_boxes[0] == [] and sample_boxes[1] == []:
            agreement = 1
        per_sample.append((folder, agreement))

    avg_agreement = np.mean([a for _, a in per_sample]) if per_sample else 1.0

    print("\nPer-sample Inter-Annotator Agreement (Hungarian Matching):")
    for folder, score in per_sample:
        print(f"  {folder:25s}  {score:.3f}")

    print(f"\nOverall Average Agreement (IoU>{iou_threshold}): {avg_agreement:.3f}")
    print(f"Compared {len(annotators)} annotators: {annotator_names}")

    print(m2/a2)
    
    return per_sample, avg_agreement


if __name__ == "__main__":
    
    MONGO_URI = os.getenv('MONGO_URI', 'mongodb://localhost:27017/')
    MONGO_DB = 'video_annotations'
    
    client = MongoClient(MONGO_URI)
    db = client[MONGO_DB]

    annotations_collection = db['annotations']
    yes_no_annotations_collection = db['yes_no_annotations']

    
    pipeline_average_time = [{"$group": {"_id": None,"avgAnnotationDuration": { "$avg": "$annotationDuration"}, "maxAnnotationDuration": { "$max": "$annotationDuration" }, "minAnnotationDuration": { "$min": "$annotationDuration" }, "medianAnnotationDuration": {"$percentile": {"input": "$annotationDuration","p": [0.5],"method": "approximate"}}}}]    
    general_statistics = list(annotations_collection.aggregate(pipeline_average_time))
    
    print('Number of total annotations:', annotations_collection.count_documents({}))
    print('')
    
    #print('Number of total annotations:', annotations_collection.count_documents({}))

    print('')
    print('Average Annotation Time (s): ', general_statistics[0]['avgAnnotationDuration']/1000)
    print('Maximum Annotation Time (s): ', general_statistics[0]['maxAnnotationDuration']/1000)
    print('Minimum Annotation Time (s): ', general_statistics[0]['minAnnotationDuration']/1000)
    print('Median Annotation Time (s): ', general_statistics[0]['medianAnnotationDuration'][0]/1000)
    print('')
    #print(A)

    pipeline_stats_per_annotator = [
        {
            "$group": {
                "_id": "$annotator_id",
                "avgMs": { "$avg": "$annotationDuration" },
                "minMs": { "$min": "$annotationDuration" },
                "maxMs": { "$max": "$annotationDuration" },
                "medianMs": {
                    "$percentile": {
                        "input": "$annotationDuration",
                        "p": [0.5],
                        "method": "approximate"
                    }
                },
                "count": { "$sum": 1 }
            }
        },
        {
            "$project": {
                "avgAnnDuration": {
                    "$round": [
                        { "$divide": ["$avgMs", 1000] },
                        2
                    ]
                },
                "minAnnDuration": {
                    "$round": [
                        { "$divide": ["$minMs", 1000] },
                        2
                    ]
                },
                "maxAnnDuration": {
                    "$round": [
                        { "$divide": ["$maxMs", 1000] },
                        2
                    ]
                },
                "medianAnnotationDuration": {
                    "$round": [
                        {
                            "$divide": [
                                { "$arrayElemAt": ["$medianMs", 0] },
                                1000
                            ]
                        },
                        2
                    ]
                },
                "count": 1
            }
        },
        {
            "$sort": {
                "count": -1
            }
        }
    ]


    stats_per_annotator = list(
        annotations_collection.aggregate(pipeline_stats_per_annotator)
    )

    print('Number of distinct annotators:', len(stats_per_annotator))
    
    for stat_per_annotator in stats_per_annotator:
        print(stat_per_annotator)

    pipeline_stats_by_globalIndex_mod3 = [
      {
        "$addFields": {
          "globalIndexMod3": { "$mod": ["$globalIndex", 3] }
        }
      },
      {
        "$group": {
          "_id": "$globalIndexMod3",
          "avgMs": { "$avg": "$annotationDuration" },
          "minMs": { "$min": "$annotationDuration" },
          "maxMs": { "$max": "$annotationDuration" },
          "medianMs": {
            "$percentile": {
              "input": "$annotationDuration",
              "p": [0.5],
              "method": "approximate"
            }
          },
          "count": { "$sum": 1 }
        }
      },
      {
        "$project": {
          "_id": 0,
          "globalIndexMod3": "$_id",
          "avgAnnDuration": {
            "$round": [{ "$divide": ["$avgMs", 1000] }, 2]
          },
          "minAnnDuration": {
            "$round": [{ "$divide": ["$minMs", 1000] }, 2]
          },
          "maxAnnDuration": {
            "$round": [{ "$divide": ["$maxMs", 1000] }, 2]
          },
          "medianAnnDuration": {
            "$round": [
              {
                "$divide": [
                  { "$arrayElemAt": ["$medianMs", 0] },
                  1000
                ]
              },
              2
            ]
          },
          "count": 1
        }
      },
      {
        "$sort": { "globalIndexMod3": 1 }
      }
    ]

    stats_per_annotator = list(
        annotations_collection.aggregate(pipeline_stats_by_globalIndex_mod3)
    )

    print('')
    print('Scattered Videos:', stats_per_annotator[1])
    print('Semi-crowded Videos:', stats_per_annotator[2])
    print('Crowded Videos:',stats_per_annotator[0])
    print('')

    #annotation_dir = "annotations"
    #compute_inter_annotator_agreement(annotation_dir)

    videoFolder = annotations_collection.aggregate([
      { "$group": { "_id": "$videoFolder" } },
      { "$sort": { "_id": 1 } }
    ])

    video_folders = sorted(set([vid['_id'] for vid in list(videoFolder)]))
    per_sample = []
    iou_threshold = 0.5
    from collections import defaultdict


    print("\nPer-sample Inter-Annotator Agreement (Hungarian Matching):")

    
    for folder in video_folders:
        #print(folder)
        test = annotations_collection.aggregate([
            {
                "$match": {
                    "videoFolder": folder,
                }
            },
            { "$unwind": "$groups" },
            {
                "$project": {
                    "_id": 0,
                    "videoFolder": 1,
                    "annotator_id": 1,
                    "annotationFrame": "$videoInfo.annotationFrame",
                    "bbox": "$groups.bbox"
                }
            }
        ])
        
        data = list(test)

        frames = defaultdict(lambda: defaultdict(list))
        
        for item in data:
            frame = item["annotationFrame"]
            annotator = item["annotator_id"]
            frames[frame][annotator].append(item["bbox"])
        
        # Final output: list[ list[ list[bbox] ] ]
        results = [
            list(annotator_groups.values())
            for frame, annotator_groups in sorted(frames.items())
        ]


        choices_frame = list(set([dat['annotationFrame'] for dat in data]))
        choices_frame.sort()

        
        for ind, result in enumerate(results):
            annotators = len(result)
            sample_boxes = []
            if annotators >= 2:
                sample_boxes = []
                for boxes in result:
                    sample_boxes.append(boxes)

    
            global m2
            global a2
            pair_agreements = []
            for i in range(len(sample_boxes)):
                for j in range(i + 1, len(sample_boxes)):
                    boxesA, boxesB = sample_boxes[i], sample_boxes[j]
                    #print(boxesA)
                    #print(boxesB)
                    matches = best_iou_matching(boxesA, boxesB, iou_threshold)
                    m2+=matches
                    avg_boxes = (len(boxesA) + len(boxesB)) / 2 if (len(boxesA) + len(boxesB)) > 0 else 1
                    pair_agreements.append(matches / avg_boxes)
                    a2+=avg_boxes
            
            #print(pair_agreements)
            agreement = np.mean(pair_agreements) if pair_agreements else 1.0
            if agreement == 0 and sample_boxes[0] == [] and sample_boxes[1] == []:
                agreement = 1
            print(f"  {folder:25s} {choices_frame[ind]:5d} {agreement:25.3f}")
            per_sample.append((folder, ind, agreement))
    
        avg_agreement = np.mean([a for _, _, a in per_sample]) if per_sample else 1.0

        
    choiced_frames = [1,21,41]
    #for folder, annFrame, score in per_sample:
    #    print(f"  {folder:25s} {choiced_frames[annFrame]:5d} {score:25.3f}")
    
    print(f"\nOverall Average Agreement (IoU>{iou_threshold}): {avg_agreement:.3f}")
        #print(f"Compared {len(annotators)} annotators: {annotator_names}")
    
        #print(m2/a2)
        

Number of total annotations: 7576


Average Annotation Time (s):  88.8024441657867
Maximum Annotation Time (s):  46324.837
Minimum Annotation Time (s):  0.163
Median Annotation Time (s):  22.654

Number of distinct annotators: 22
{'_id': '45efce7c-be08-4706-9de0-de6ee82a7520', 'count': 2550, 'avgAnnDuration': 73.69, 'minAnnDuration': 0.3, 'maxAnnDuration': 13000.22, 'medianAnnotationDuration': 29.34}
{'_id': '1359b1ec-c92f-4f51-9352-0ce981f20bf3', 'count': 1350, 'avgAnnDuration': 231.24, 'minAnnDuration': 0.5, 'maxAnnDuration': 46324.84, 'medianAnnotationDuration': 112.98}
{'_id': '70bdd7d2-b639-4328-b4be-5c62e8b6cb22', 'count': 900, 'avgAnnDuration': 12.58, 'minAnnDuration': 0.16, 'maxAnnDuration': 347.95, 'medianAnnotationDuration': 8.23}
{'_id': '93037c9e-9ad4-4e9f-92b5-9b7f1c13ea44', 'count': 555, 'avgAnnDuration': 19.86, 'minAnnDuration': 0.74, 'maxAnnDuration': 566.73, 'medianAnnotationDuration': 11.85}
{'_id': 'c2e699e0-103e-4e7a-a1ae-35dbb3aa3419', 'count': 540, 'avgAnnDuration

## FINEGRAINED ANNOTATIONS

In [22]:
import os
import json
import numpy as np

from scipy.optimize import linear_sum_assignment
from pymongo import MongoClient

m2 = 0
a2 = 0

def iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    interW = max(0, xB - xA)
    interH = max(0, yB - yA)
    interArea = interW * interH

    if interArea == 0:
        return 0.0

    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    unionArea = boxAArea + boxBArea - interArea
    return interArea / unionArea


def best_iou_matching(boxesA, boxesB, threshold=0.4):
    if not boxesA or not boxesB:
        return 0

    nA, nB = len(boxesA), len(boxesB)
    iou_matrix = np.zeros((nA, nB))

    for i in range(nA):
        for j in range(nB):
            iou_matrix[i, j] = iou(boxesA[i], boxesB[j])

    row_ind, col_ind = linear_sum_assignment(-iou_matrix)
    matches = sum(iou_matrix[i, j] >= threshold for i, j in zip(row_ind, col_ind))
    return matches


def compute_inter_annotator_agreement(annotation_dir, iou_threshold=0.4):

    annotators = []
    annotator_names = []
    for file in os.listdir(annotation_dir):
        if file.endswith(".json"):
            with open(os.path.join(annotation_dir, file), "r") as f:
                data = json.load(f)
                annotators.append(data["annotations"])
                annotator_names.append(file)
                
    if len(annotators) < 2:
        raise ValueError("Need at least two annotators to compute agreement.")

    video_folders = sorted(set(a["videoFolder"] for ann in annotators for a in ann))
    print(video_folders)
    per_sample = []

    for folder in video_folders:
        sample_boxes = []
        for ann in annotators:
            boxes = []
            for entry in ann:
                if entry["videoFolder"] == folder:
                    boxes = [g["bbox"] for g in entry["groups"] if g["confidence"] >= 1]
                    break
            sample_boxes.append(boxes)

        #print(sample_boxes)

        global m2
        global a2
        pair_agreements = []
        for i in range(len(sample_boxes)):
            for j in range(i + 1, len(sample_boxes)):
                boxesA, boxesB = sample_boxes[i], sample_boxes[j]
                #print(boxesA)
                #print(boxesB)
                matches = best_iou_matching(boxesA, boxesB, iou_threshold)
                m2+=matches
                avg_boxes = (len(boxesA) + len(boxesB)) / 2 if (len(boxesA) + len(boxesB)) > 0 else 1
                pair_agreements.append(matches / avg_boxes)
                a2+=avg_boxes
        
        #print(pair_agreements)
        agreement = np.mean(pair_agreements) if pair_agreements else 1.0
        if agreement == 0 and sample_boxes[0] == [] and sample_boxes[1] == []:
            agreement = 1
        per_sample.append((folder, agreement))

    avg_agreement = np.mean([a for _, a in per_sample]) if per_sample else 1.0

    print("\nPer-sample Inter-Annotator Agreement (Hungarian Matching):")
    for folder, score in per_sample:
        print(f"  {folder:25s}  {score:.3f}")

    print(f"\nOverall Average Agreement (IoU>{iou_threshold}): {avg_agreement:.3f}")
    print(f"Compared {len(annotators)} annotators: {annotator_names}")

    print(m2/a2)
    
    return per_sample, avg_agreement


if __name__ == "__main__":
    
    MONGO_URI = os.getenv('MONGO_URI', 'mongodb://localhost:27017/')
    MONGO_DB = 'video_annotations'
    
    client = MongoClient(MONGO_URI)
    db = client[MONGO_DB]

    annotations_collection = db['finegrained_annotations']
    #annotations_collection = db['annotations']
    
    #yes_no_annotations_collection = db['yes_no_annotations']

    
    pipeline_average_time = [{"$group": {"_id": None,"avgAnnotationDuration": { "$avg": "$annotationDuration"}, "maxAnnotationDuration": { "$max": "$annotationDuration" }, "minAnnotationDuration": { "$min": "$annotationDuration" }, "medianAnnotationDuration": {"$percentile": {"input": "$annotationDuration","p": [0.5],"method": "approximate"}}}}]    
    general_statistics = list(annotations_collection.aggregate(pipeline_average_time))
    
    print('Number of total annotations:', annotations_collection.count_documents({}))
    print('')
    
    #print('Number of total annotations:', annotations_collection.count_documents({}))

    print('')
    print('Average Annotation Time (s): ', general_statistics[0]['avgAnnotationDuration']/1000)
    print('Maximum Annotation Time (s): ', general_statistics[0]['maxAnnotationDuration']/1000)
    print('Minimum Annotation Time (s): ', general_statistics[0]['minAnnotationDuration']/1000)
    print('Median Annotation Time (s): ', general_statistics[0]['medianAnnotationDuration'][0]/1000)
    print('')

    
    pipeline_stats_per_annotator = [
        {
            "$group": {
                "_id": "$annotator_id",
                "avgMs": { "$avg": "$annotationDuration" },
                "minMs": { "$min": "$annotationDuration" },
                "maxMs": { "$max": "$annotationDuration" },
                "medianMs": {
                    "$percentile": {
                        "input": "$annotationDuration",
                        "p": [0.5],
                        "method": "approximate"
                    }
                },
                "count": { "$sum": 1 }
            }
        },
        {
            "$project": {
                "avgAnnDuration": {
                    "$round": [
                        { "$divide": ["$avgMs", 1000] },
                        2
                    ]
                },
                "minAnnDuration": {
                    "$round": [
                        { "$divide": ["$minMs", 1000] },
                        2
                    ]
                },
                "maxAnnDuration": {
                    "$round": [
                        { "$divide": ["$maxMs", 1000] },
                        2
                    ]
                },
                "medianAnnotationDuration": {
                    "$round": [
                        {
                            "$divide": [
                                { "$arrayElemAt": ["$medianMs", 0] },
                                1000
                            ]
                        },
                        2
                    ]
                },
                "count": 1
            }
        },
        {
            "$sort": {
                "count": -1
            }
        }
    ]


    stats_per_annotator = list(
        annotations_collection.aggregate(pipeline_stats_per_annotator)
    )

    print('Number of distinct annotators:', len(stats_per_annotator))
    
    for stat_per_annotator in stats_per_annotator:
        print(stat_per_annotator)

    pipeline_stats_by_globalIndex_mod3 = [
      {
        "$addFields": {
          "globalIndexMod3": { "$mod": ["$globalIndex", 3] }
        }
      },
      {
        "$group": {
          "_id": "$globalIndexMod3",
          "avgMs": { "$avg": "$annotationDuration" },
          "minMs": { "$min": "$annotationDuration" },
          "maxMs": { "$max": "$annotationDuration" },
          "medianMs": {
            "$percentile": {
              "input": "$annotationDuration",
              "p": [0.5],
              "method": "approximate"
            }
          },
          "count": { "$sum": 1 }
        }
      },
      {
        "$project": {
          "_id": 0,
          "globalIndexMod3": "$_id",
          "avgAnnDuration": {
            "$round": [{ "$divide": ["$avgMs", 1000] }, 2]
          },
          "minAnnDuration": {
            "$round": [{ "$divide": ["$minMs", 1000] }, 2]
          },
          "maxAnnDuration": {
            "$round": [{ "$divide": ["$maxMs", 1000] }, 2]
          },
          "medianAnnDuration": {
            "$round": [
              {
                "$divide": [
                  { "$arrayElemAt": ["$medianMs", 0] },
                  1000
                ]
              },
              2
            ]
          },
          "count": 1
        }
      },
      {
        "$sort": { "globalIndexMod3": 1 }
      }
    ]

    stats_per_annotator = list(
        annotations_collection.aggregate(pipeline_stats_by_globalIndex_mod3)
    )

    print('')
    print('Scattered Videos:', stats_per_annotator[1])
    print('Semi-crowded Videos:', stats_per_annotator[2])
    print('Crowded Videos:',stats_per_annotator[0])
    print('')
    
    #print(A)

Number of total annotations: 8100


Average Annotation Time (s):  156.61648308641975
Maximum Annotation Time (s):  181982.355
Minimum Annotation Time (s):  0.05
Median Annotation Time (s):  21.162875

Number of distinct annotators: 22
{'_id': '45efce7c-be08-4706-9de0-de6ee82a7520', 'count': 2613, 'avgAnnDuration': 98.89, 'minAnnDuration': 1.08, 'maxAnnDuration': 44736.67, 'medianAnnotationDuration': 23.8}
{'_id': '5c662e2e-1601-49f5-9b3b-374d26b2b6ec', 'count': 1502, 'avgAnnDuration': 30.1, 'minAnnDuration': 0.53, 'maxAnnDuration': 3254.4, 'medianAnnotationDuration': 14.98}
{'_id': '8c662e2e-1601-49f5-9b3b-374d26b2b6ec', 'count': 1120, 'avgAnnDuration': 99.17, 'minAnnDuration': 1.08, 'maxAnnDuration': 44736.67, 'medianAnnotationDuration': 20.42}
{'_id': '1359b1ec-c92f-4f51-9352-0ce981f20bf3', 'count': 915, 'avgAnnDuration': 389.92, 'minAnnDuration': 1.7, 'maxAnnDuration': 45562.6, 'medianAnnotationDuration': 92.75}
{'_id': '70bdd7d2-b639-4328-b4be-5c62e8b6cb22', 'count': 568, 'avgAnnDu

## DELETION / MODIFICATION RATE (finegrained)

In [23]:
import os
import json
from collections import defaultdict
import numpy as np
from scipy.optimize import linear_sum_assignment
from pymongo import MongoClient

MONGO_URI = os.getenv('MONGO_URI', 'mongodb://localhost:27017/')
MONGO_DB = 'video_annotations'

client = MongoClient(MONGO_URI)
db = client[MONGO_DB]
finegrained_collection = db['finegrained_annotations']

# SAM3 pre-computed detections, keyed "{video_index}_{annotatedFrame}" -- this is
# what finegrained annotators start from (see app.py finegrained_detect_frame).
for _candidate in ("../detections_cache_sam3.json", "detections_cache_sam3.json",
                    "../detections_cache_sam3_v1.json", "detections_cache_sam3_v1.json"):
    if os.path.exists(_candidate):
        with open(_candidate) as _f:
            DETECTIONS_CACHE = json.load(_f)
        break
else:
    raise FileNotFoundError("Could not find detections_cache_sam3(.json/_v1.json) relative to cwd")


def deletion_rate_qc(collection, label, min_annotations=5):
    """
    Deletion rate = numberOfDeletedGroups / (numberOfGroups + numberOfDeletedGroups):
    the fraction of all boxes an annotator was presented with that they ultimately
    discarded -- i.e. how much of the SAM3 pre-computed detection got rejected.
    """
    pipeline_base = [
        {"$addFields": {"presented": {"$add": ["$numberOfGroups", "$numberOfDeletedGroups"]}}},
        {"$addFields": {
            "delRate": {
                "$cond": [{"$gt": ["$presented", 0]},
                          {"$divide": ["$numberOfDeletedGroups", "$presented"]}, 0]
            }
        }}
    ]

    overall = list(collection.aggregate(pipeline_base + [
        {"$group": {
            "_id": None,
            "n": {"$sum": 1},
            "avgDelRate": {"$avg": "$delRate"},
            "docsWithAnyDeletion": {"$sum": {"$cond": [{"$gt": ["$numberOfDeletedGroups", 0]}, 1, 0]}},
            "avgPresented": {"$avg": "$presented"},
            "avgDeleted": {"$avg": "$numberOfDeletedGroups"}
        }}
    ]))[0]

    print(f"=== {label}: deletion rate QC ===")
    print(f"Total annotations: {overall['n']}")
    print(f"Avg boxes presented per annotation: {overall['avgPresented']:.2f}   avg deleted: {overall['avgDeleted']:.2f}")
    print(f"Docs with >=1 deletion: {overall['docsWithAnyDeletion']} ({100*overall['docsWithAnyDeletion']/overall['n']:.1f}%)")
    print(f"Avg per-doc deletion rate: {overall['avgDelRate']*100:.1f}%")

    per_annotator = list(collection.aggregate(pipeline_base + [
        {"$group": {
            "_id": "$annotator_id",
            "n": {"$sum": 1},
            "avgDelRate": {"$avg": "$delRate"},
            "avgPresented": {"$avg": "$presented"}
        }},
        {"$match": {"n": {"$gte": min_annotations}}},
        {"$project": {"n": 1, "avgPresented": {"$round": ["$avgPresented", 2]},
                      "delRatePct": {"$round": [{"$multiply": ["$avgDelRate", 100]}, 1]}}},
        {"$sort": {"delRatePct": -1}}
    ]))
    print(f"\nPer-annotator deletion rate (n>={min_annotations}, worst first):")
    for a in per_annotator:
        print(f"  {a['_id']:38s} n={a['n']:5d}  avgPresented={a['avgPresented']:6.2f}  delRate={a['delRatePct']:5.1f}%")

    by_crowd = list(collection.aggregate(pipeline_base + [
        {"$addFields": {"globalIndexMod3": {"$mod": ["$globalIndex", 3]}}},
        {"$group": {"_id": "$globalIndexMod3", "n": {"$sum": 1}, "avgDelRate": {"$avg": "$delRate"}}},
        {"$sort": {"_id": 1}}
    ]))
    bucket_names = {1: "Scattered", 2: "Semi-crowded", 0: "Crowded"}
    print(f"\nDeletion rate by crowd-density bucket:")
    for b in by_crowd:
        print(f"  {bucket_names.get(b['_id'], b['_id']):15s} n={b['n']:5d}  avgDelRate={b['avgDelRate']*100:.1f}%")
    print()


deletion_rate_qc(finegrained_collection, 'FINEGRAINED')


def iou_xyxy(a, b):
    xA, yA = max(a[0], b[0]), max(a[1], b[1])
    xB, yB = min(a[2], b[2]), min(a[3], b[3])
    interW, interH = max(0, xB - xA), max(0, yB - yA)
    inter = interW * interH
    if inter == 0:
        return 0.0
    areaA = (a[2] - a[0]) * (a[3] - a[1])
    areaB = (b[2] - b[0]) * (b[3] - b[1])
    return inter / (areaA + areaB - inter)


def finegrained_modification_qc(min_annotations=5, iou_low=0.3, iou_high=0.9):
    """
    Finegrained groups carry no modificationHistory, so "modified" is inferred by
    matching each kept (non-deleted) box back to the SAM3 detection it started
    from (via app.py's own cache_key = "{video_index}_{annotatedFrame}") using
    Hungarian IoU matching, then classifying by best-match IoU:
      - iou < iou_low:            "added"       -- no matching SAM3 box, drawn from scratch
      - iou_low <= iou < iou_high: "modified"    -- kept the SAM3 box but moved/resized it
      - iou >= iou_high:           "unmodified"  -- accepted the SAM3 box essentially as-is
    """
    docs = list(finegrained_collection.find({}, {
        "videoFolder": 1, "annotatedFrame": 1, "annotator_id": 1, "globalIndex": 1,
        "groups.bbox": 1, "groups.isDeleted": 1
    }))

    cat_counts = defaultdict(int)
    per_annotator = defaultdict(lambda: defaultdict(int))
    per_bucket = defaultdict(lambda: defaultdict(int))
    mod_ious = []
    docs_no_cache = 0

    for d in docs:
        video_index = int(d["videoFolder"].split("_")[-1])
        cache_key = f"{video_index}_{d['annotatedFrame']}"
        kept_boxes = [g["bbox"] for g in d["groups"] if not g.get("isDeleted", False)]
        if not kept_boxes:
            continue

        aid, bucket = d["annotator_id"], d["globalIndex"] % 3
        sam3_dets = DETECTIONS_CACHE.get(cache_key, {}).get("detections", [])

        if not sam3_dets:
            docs_no_cache += 1
            best_iou = [0.0] * len(kept_boxes)
        else:
            sam3_boxes = [det["bbox"] for det in sam3_dets]
            nA, nB = len(kept_boxes), len(sam3_boxes)
            iou_matrix = np.zeros((nA, nB))
            for i in range(nA):
                for j in range(nB):
                    iou_matrix[i, j] = iou_xyxy(kept_boxes[i], sam3_boxes[j])
            row_ind, col_ind = linear_sum_assignment(-iou_matrix)
            best_iou = [0.0] * nA
            for i, j in zip(row_ind, col_ind):
                best_iou[i] = iou_matrix[i, j]

        for biou in best_iou:
            per_annotator[aid]["total"] += 1
            per_bucket[bucket]["total"] += 1
            if biou < iou_low:
                cat = "added"
            elif biou < iou_high:
                cat = "modified"
                mod_ious.append(biou)
            else:
                cat = "unmodified"
            cat_counts[cat] += 1
            per_annotator[aid][cat] += 1
            per_bucket[bucket][cat] += 1

    total = sum(cat_counts.values())
    print("=== FINEGRAINED: box modification rate (vs. SAM3 pre-computed detection) ===")
    print(f"Docs with no cached SAM3 detections for their frame: {docs_no_cache}")
    print(f"Total kept boxes: {total}")
    for cat in ("unmodified", "modified", "added"):
        v = cat_counts[cat]
        print(f"  {cat:12s} {v:6d} ({100*v/total:.1f}%)")
    if mod_ious:
        print(f"Avg IoU(SAM3 -> final) for modified boxes: {np.mean(mod_ious):.3f}  median={np.median(mod_ious):.3f}")

    rows = [(aid, c["total"], c["modified"], 100 * c["modified"] / c["total"])
            for aid, c in per_annotator.items() if c["total"] >= min_annotations]
    rows.sort(key=lambda r: -r[3])
    print(f"\nPer-annotator modification rate (n>={min_annotations}, worst first):")
    for aid, n, m, pct in rows:
        print(f"  {aid:38s} n={n:6d}  modified={m:5d} ({pct:5.1f}%)")

    bucket_names = {1: "Scattered", 2: "Semi-crowded", 0: "Crowded"}
    print(f"\nModification rate by crowd-density bucket:")
    for bkt in sorted(per_bucket):
        c = per_bucket[bkt]
        print(f"  {bucket_names[bkt]:15s} n={c['total']:6d}  modified={c['modified']:5d} ({100*c['modified']/c['total']:.1f}%)")


finegrained_modification_qc()


=== FINEGRAINED: deletion rate QC ===
Total annotations: 8100
Avg boxes presented per annotation: 15.02   avg deleted: 0.66
Docs with >=1 deletion: 2179 (26.9%)
Avg per-doc deletion rate: 3.2%

Per-annotator deletion rate (n>=5, worst first):
  ab438db2-26fd-47be-bbca-f1cc0aae6807   n=   15  avgPresented= 14.73  delRate= 23.5%
  1040bec8-d619-4c5a-ad8a-2c6f07597829   n=   63  avgPresented= 15.83  delRate= 14.1%
  a1006df4-3654-444a-90fa-0d0c7455b191   n=    5  avgPresented= 24.00  delRate= 10.7%
  f3771ee3-7a60-4a6f-b914-1aef42c9203a   n=   28  avgPresented= 15.68  delRate=  9.7%
  1359b1ec-c92f-4f51-9352-0ce981f20bf3   n=  915  avgPresented= 15.75  delRate=  8.8%
  fab80631-7069-4f8f-8181-702870f9e3f8   n=   15  avgPresented= 18.20  delRate=  7.8%
  c2f49414-8d0a-4dfa-a9df-254014aae30b   n=   15  avgPresented= 15.27  delRate=  5.9%
  405937c9-c817-4dd0-ba8b-d2cc7d119074   n=   15  avgPresented= 16.80  delRate=  5.4%
  b0df9268-c7d6-46b4-8e39-5bc5af7ab54d   n=   14  avgPresented= 20.21

## GROUP-COUNT CORRELATION (finegrained)

In [24]:
import os
from collections import defaultdict
import numpy as np
from scipy.stats import pearsonr, spearmanr
from pymongo import MongoClient

MONGO_URI = os.getenv('MONGO_URI', 'mongodb://localhost:27017/')
MONGO_DB = 'video_annotations'

client = MongoClient(MONGO_URI)
db = client[MONGO_DB]
finegrained_collection = db['finegrained_annotations']

# Distinct group count per annotation = number of unique groupId values among kept
# (non-deleted) boxes, excluding groupId == -1 ("individual", not a group). This is
# NOT the same as the doc-level `numberOfGroups` field, which counts total boxes.

docs = list(finegrained_collection.find({}, {
    "videoFolder": 1, "annotatedFrame": 1, "annotator_id": 1, "globalIndex": 1,
    "groups.groupId": 1, "groups.isDeleted": 1
}))

by_frame = defaultdict(dict)  # (videoFolder, annotatedFrame) -> {annotator_id: numGroups}
bucket_of = {}

for d in docs:
    kept_group_ids = {g["groupId"] for g in d["groups"]
                       if not g.get("isDeleted", False) and g["groupId"] != -1}
    key = (d["videoFolder"], d["annotatedFrame"])
    by_frame[key][d["annotator_id"]] = len(kept_group_ids)
    bucket_of[key] = d["globalIndex"] % 3

pairsA, pairsB, bucket_pair = [], [], []
for key, annmap in by_frame.items():
    if len(annmap) < 2:
        continue
    vals = list(annmap.values())
    for i in range(len(vals)):
        for j in range(i + 1, len(vals)):
            pairsA.append(vals[i])
            pairsB.append(vals[j])
            bucket_pair.append(bucket_of[key])

pairsA, pairsB = np.array(pairsA), np.array(pairsB)
r, rp = pearsonr(pairsA, pairsB)
rho, rhop = spearmanr(pairsA, pairsB)

print("=== FINEGRAINED: group-count correlation (distinct groupId per annotator pair) ===")
print(f"Frames with >=2 annotators: {len(by_frame) - sum(1 for a in by_frame.values() if len(a) < 2)}")
print(f"Pooled annotator-pair samples: {len(pairsA)}")
print(f"Pearson r={r:.3f} (p={rp:.2e})   Spearman rho={rho:.3f} (p={rhop:.2e})")
print(f"MAE: {np.mean(np.abs(pairsA - pairsB)):.3f}   Exact match: {100*np.mean(pairsA == pairsB):.1f}%")
print(f"Group-count range: [{min(pairsA.min(), pairsB.min())}-{max(pairsA.max(), pairsB.max())}]  "
      f"mean A={pairsA.mean():.2f} B={pairsB.mean():.2f}")

bucket_pair = np.array(bucket_pair)
bucket_names = {1: "Scattered", 2: "Semi-crowded", 0: "Crowded"}
print("\nBy crowd-density bucket:")
for b in sorted(set(bucket_pair)):
    mask = bucket_pair == b
    a, bb = pairsA[mask], pairsB[mask]
    r_b, _ = pearsonr(a, bb)
    rho_b, _ = spearmanr(a, bb)
    print(f"  {bucket_names[b]:15s} n={mask.sum():5d}  r={r_b:.3f}  rho={rho_b:.3f}  MAE={np.mean(np.abs(a-bb)):.3f}")


=== FINEGRAINED: group-count correlation (distinct groupId per annotator pair) ===
Frames with >=2 annotators: 2700
Pooled annotator-pair samples: 8100
Pearson r=0.966 (p=0.00e+00)   Spearman rho=0.969 (p=0.00e+00)
MAE: 0.222   Exact match: 84.6%
Group-count range: [0-25]  mean A=3.07 B=3.00

By crowd-density bucket:
  Crowded         n= 2700  r=0.938  rho=0.938  MAE=0.386
  Scattered       n= 2700  r=0.964  rho=0.974  MAE=0.064
  Semi-crowded    n= 2700  r=0.968  rho=0.968  MAE=0.217


## GROUP-COUNT CORRELATION (coarse)

In [25]:
import os
from collections import defaultdict
import numpy as np
from scipy.stats import pearsonr, spearmanr
from pymongo import MongoClient

MONGO_URI = os.getenv('MONGO_URI', 'mongodb://localhost:27017/')
MONGO_DB = 'video_annotations'

client = MongoClient(MONGO_URI)
db = client[MONGO_DB]
coarse_collection = db['annotations']

# Coarse groups have no clustering step (unlike finegrained's groupId) -- each kept
# entry in `groups[]` IS one drawn group, so the doc-level `numberOfGroups` field
# already is the group count.

docs = list(coarse_collection.find({}, {
    "videoFolder": 1, "videoInfo.annotationFrame": 1, "annotator_id": 1,
    "globalIndex": 1, "numberOfGroups": 1
}))

by_frame = defaultdict(dict)  # (videoFolder, annotationFrame) -> {annotator_id: numGroups}
bucket_of = {}

for d in docs:
    key = (d["videoFolder"], d["videoInfo"]["annotationFrame"])
    by_frame[key][d["annotator_id"]] = d["numberOfGroups"]
    bucket_of[key] = d["globalIndex"] % 3

pairsA, pairsB, bucket_pair = [], [], []
for key, annmap in by_frame.items():
    if len(annmap) < 2:
        continue
    vals = list(annmap.values())
    for i in range(len(vals)):
        for j in range(i + 1, len(vals)):
            pairsA.append(vals[i])
            pairsB.append(vals[j])
            bucket_pair.append(bucket_of[key])

pairsA, pairsB = np.array(pairsA), np.array(pairsB)
r, rp = pearsonr(pairsA, pairsB)
rho, rhop = spearmanr(pairsA, pairsB)

print("=== COARSE: group-count correlation (numberOfGroups per annotator pair) ===")
print(f"Frames with >=2 annotators: {sum(1 for a in by_frame.values() if len(a) >= 2)}")
print(f"Pooled annotator-pair samples: {len(pairsA)}")
print(f"Pearson r={r:.3f} (p={rp:.2e})   Spearman rho={rho:.3f} (p={rhop:.2e})")
print(f"MAE: {np.mean(np.abs(pairsA - pairsB)):.3f}   Exact match: {100*np.mean(pairsA == pairsB):.1f}%")
print(f"Group-count range: [{min(pairsA.min(), pairsB.min())}-{max(pairsA.max(), pairsB.max())}]  "
      f"mean A={pairsA.mean():.2f} B={pairsB.mean():.2f}")

bucket_pair = np.array(bucket_pair)
bucket_names = {1: "Scattered", 2: "Semi-crowded", 0: "Crowded"}
print("\nBy crowd-density bucket:")
for b in sorted(set(bucket_pair)):
    mask = bucket_pair == b
    a, bb = pairsA[mask], pairsB[mask]
    r_b, _ = pearsonr(a, bb)
    rho_b, _ = spearmanr(a, bb)
    print(f"  {bucket_names[b]:15s} n={mask.sum():5d}  r={r_b:.3f}  rho={rho_b:.3f}  MAE={np.mean(np.abs(a-bb)):.3f}")


=== COARSE: group-count correlation (numberOfGroups per annotator pair) ===
Frames with >=2 annotators: 2699
Pooled annotator-pair samples: 7067
Pearson r=0.649 (p=0.00e+00)   Spearman rho=0.736 (p=0.00e+00)
MAE: 1.081   Exact match: 48.0%
Group-count range: [0-25]  mean A=1.96 B=2.35

By crowd-density bucket:
  Crowded         n= 2354  r=0.481  rho=0.489  MAE=1.745
  Scattered       n= 2358  r=0.714  rho=0.725  MAE=0.336
  Semi-crowded    n= 2355  r=0.618  rho=0.689  MAE=1.163


## GROUP-IDENTITY AGREEMENT (IoU-matched, finegrained)

In [ ]:
import os
from collections import defaultdict, Counter
from math import comb
import numpy as np
from scipy.optimize import linear_sum_assignment
from pymongo import MongoClient

MONGO_URI = os.getenv('MONGO_URI', 'mongodb://localhost:27017/')
MONGO_DB = 'video_annotations'

client = MongoClient(MONGO_URI)
db = client[MONGO_DB]
finegrained_collection = db['finegrained_annotations']

# The group-count correlation above only compares HOW MANY groups each annotator
# found -- it can't tell whether they grouped the SAME people. This section fixes
# that: it uses IoU (the same Hungarian-matching approach as the original coarse
# `compute_inter_annotator_agreement`/box-matching code) to establish which boxes
# from annotator A and annotator B are the same physical person, then checks
# whether the matched people's group memberships actually agree.

IOU_THRESHOLD = 0.5


def iou(a, b):
    xA, yA = max(a[0], b[0]), max(a[1], b[1])
    xB, yB = min(a[2], b[2]), min(a[3], b[3])
    interW, interH = max(0, xB - xA), max(0, yB - yA)
    inter = interW * interH
    if inter == 0:
        return 0.0
    areaA = (a[2] - a[0]) * (a[3] - a[1])
    areaB = (b[2] - b[0]) * (b[3] - b[1])
    return inter / (areaA + areaB - inter)


def adjusted_rand_index(labelsA, labelsB):
    """Chance-corrected pairwise co-clustering agreement (standard ARI formula)."""
    n = len(labelsA)
    if n < 2:
        return None
    contingency = Counter(zip(labelsA, labelsB))
    sum_comb_c = sum(comb(v, 2) for v in contingency.values())
    sum_comb_rows = sum(comb(v, 2) for v in Counter(labelsA).values())
    sum_comb_cols = sum(comb(v, 2) for v in Counter(labelsB).values())
    total_comb = comb(n, 2)
    expected_index = (sum_comb_rows * sum_comb_cols) / total_comb
    max_index = 0.5 * (sum_comb_rows + sum_comb_cols)
    denom = max_index - expected_index
    return 1.0 if denom == 0 else (sum_comb_c - expected_index) / denom


docs = list(finegrained_collection.find({}, {
    "videoFolder": 1, "annotatedFrame": 1, "annotator_id": 1,
    "groups.bbox": 1, "groups.groupId": 1, "groups.isDeleted": 1
}))

by_frame = defaultdict(dict)  # (videoFolder, frame) -> {annotator_id: [(bbox, groupId), ...]}
for d in docs:
    key = (d["videoFolder"], d["annotatedFrame"])
    entries = [(g["bbox"], g["groupId"]) for g in d["groups"] if not g.get("isDeleted", False)]
    by_frame[key][d["annotator_id"]] = entries

box_match_rates = []       # matches / avg(#boxes) per annotator-pair-per-frame -- same
                            # metric family as the original pasted box-matching code
rand_scores = []           # raw pairwise co-clustering agreement (Rand Index) over matched people
ari_scores = []            # chance-corrected version (Adjusted Rand Index)

for key, annmap in by_frame.items():
    if len(annmap) < 2:
        continue
    annotators = list(annmap.items())
    for i in range(len(annotators)):
        for j in range(i + 1, len(annotators)):
            (_, entriesA), (_, entriesB) = annotators[i], annotators[j]
            if not entriesA or not entriesB:
                continue
            boxesA, boxesB = [e[0] for e in entriesA], [e[0] for e in entriesB]
            nA, nB = len(boxesA), len(boxesB)
            iou_matrix = np.zeros((nA, nB))
            for a in range(nA):
                for b in range(nB):
                    iou_matrix[a, b] = iou(boxesA[a], boxesB[b])
            row_ind, col_ind = linear_sum_assignment(-iou_matrix)
            matched = [(entriesA[a][1], entriesB[b][1]) for a, b in zip(row_ind, col_ind)
                       if iou_matrix[a, b] >= IOU_THRESHOLD]

            avg_boxes = (nA + nB) / 2
            box_match_rates.append(len(matched) / avg_boxes if avg_boxes > 0 else 1.0)

            M = len(matched)
            if M < 2:
                continue

            # -1 ("individual") must NOT be treated as one shared group across
            # different people -- give every individual a unique per-position label
            # so two unrelated solo people never count as "grouped together".
            labelsA = [g[0] if g[0] != -1 else f"solo_{k}" for k, g in enumerate(matched)]
            labelsB = [g[1] if g[1] != -1 else f"solo_{k}" for k, g in enumerate(matched)]

            agree = sum(
                ((labelsA[k] == labelsA[l]) == (labelsB[k] == labelsB[l]))
                for k in range(M) for l in range(k + 1, M)
            )
            total = M * (M - 1) // 2
            rand_scores.append(agree / total)

            ari = adjusted_rand_index(labelsA, labelsB)
            if ari is not None:
                ari_scores.append(ari)

print("=== FINEGRAINED: group-identity agreement (IoU-matched people) ===")
print(f"Annotator-pair-per-frame samples (box matching): {len(box_match_rates)}")
print(f"Avg box IoU-match rate (matches / avg #boxes, IoU>={IOU_THRESHOLD}): {np.mean(box_match_rates):.3f}")
print("  -- this is the SAME metric the original coarse box-matching code computes:")
print("     it only checks whether the same physical people were detected, not whether")
print("     they were grouped the same way.")

print(f"\nSamples with >=2 IoU-matched people (needed to compare grouping): {len(rand_scores)}")
print(f"Avg raw pairwise co-clustering agreement (Rand Index): {np.mean(rand_scores):.3f}")
print("  -- inflated: most groups are solo (size 1, see GROUP SIZE DISTRIBUTION), so most")
print("     person-pairs are trivially 'not grouped together' in both annotations, which")
print("     counts as 'agreement' even though it says nothing about real clustering skill.")

print(f"\nAvg Adjusted Rand Index (chance-corrected): {np.mean(ari_scores):.3f}  median={np.median(ari_scores):.3f}")
print("  -- this is the honest number: how much annotators agree on WHO is grouped with")
print("     WHOM, after removing the credit for trivially agreeing two solo people aren't grouped.")

print(f"\nSummary: box-match rate {np.mean(box_match_rates):.3f}  ->  raw co-clustering {np.mean(rand_scores):.3f}"
      f"  ->  chance-corrected (ARI) {np.mean(ari_scores):.3f}")
